In [1]:
%pip install ipykernel ipywidgets scikit-learn transformers sentencepiece onnxruntime huggingface-hub torch


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Кластеризация новостей с BGE-M3 и локальным reranker

Срез **22–23 октября 2025 (UTC)**. Для каждой новости ищется настраиваемое число наиболее близких по cosine кластеров (по умолчанию 10), активных за последние 24 часа. Локальный ONNX INT8 reranker выбирает лучший среди кандидатов, прошедших cosine-порог. Если ни один не прошел порог, создается новый кластер.

Запускайте ячейки сверху вниз. Эмбеддинги кэшируются в `outputs/cluster_explorer/`; сохраненные предсказания ruBERT используются повторно. Настройки доступны в виджетах. Кнопка HTML создает автономный файл с поиском, фильтром по источнику и метке, сортировкой и сворачиванием кластеров.

Первая ячейка устанавливает зависимости прямо в окружение ядра ноутбука.


In [2]:
from pathlib import Path
import sys, json, hashlib, time, html, importlib
from datetime import timedelta
import numpy as np
import ipywidgets as W
from IPython.display import display, HTML, clear_output

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "core/news_data.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите ноутбук из ml или ml/notebooks")
sys.path.insert(0, str(ROOT))
from core.news_data import load_slice, load_predictions, parse_datetime
import core.news_cluster_reranker as _cluster_models
_cluster_models = importlib.reload(_cluster_models)
EMBEDDING_MODEL = _cluster_models.EMBEDDING_MODEL
RERANKER_MODEL = _cluster_models.RERANKER_MODEL
OnnxEmbedder = _cluster_models.OnnxEmbedder
LocalReranker = _cluster_models.LocalReranker
cluster_posts = _cluster_models.cluster_posts

START = parse_datetime("2025-10-22T00:00:00Z")
END = parse_datetime("2025-10-24T00:00:00Z")
OUTPUT = ROOT / "outputs"
PREDICTIONS = OUTPUT / "news_predictions_2025-10-22_2025-10-23.jsonl"
CACHE_DIR = OUTPUT / "cluster_explorer"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
clusters = None
run_info = None
records = None
vectors = None
vector_cache_key = None
DEVICE = "cpu"  # Готовые INT8 ONNX-модели используют CPUExecutionProvider.
print(f"ONNX Runtime: {DEVICE}")


ONNX Runtime: cpu


## Настройки и пересчет

Cosine используется для отбора активных кластеров и проверки порога. Число кандидатов для reranker настраивается отдельно (по умолчанию 10, например 16). Все прошедшие порог пары из этого списка передаются локальному reranker. Окно поиска, порог, длина текста и размеры батчей можно менять перед пересчетом. Модели загружаются локально из `final/models/`.


In [3]:
from core.pipeline import get_vectors

def get_data_and_vectors(chars, embedding_batch_size, model_name):
    global records, vectors
    raw, _ = load_slice(ROOT / "data/raw", START, END)
    retained, counts, _ = load_predictions(raw, PREDICTIONS)
    if not retained:
        raise ValueError("Нет новостей после фильтрации ruBERT")
    records = retained
    vectors = get_vectors(records, model_name, CACHE_DIR, chars, embedding_batch_size)
    return records, vectors, counts


cosine_threshold = W.FloatSlider(value=_cluster_models.DEFAULT_COSINE_THRESHOLD, min=.30, max=1., step=.01, readout_format=".2f",
                                 description="Cosine порог:", continuous_update=False)
window_hours = W.BoundedIntText(value=24, min=1, max=168, description="Окно, ч:")
embedding_chars = W.BoundedIntText(value=1200, min=0, max=20000, description="Символов:",
                                   tooltip="0 означает полный очищенный текст")
embedding_batch_size = W.BoundedIntText(value=16, min=1, max=128, description="Batch BGE:")
reranker_candidates = W.BoundedIntText(value=8, min=1, max=50, description="Кандидатов reranker:")
reranker_batch_size = W.BoundedIntText(value=8, min=1, max=64, description="Batch reranker:")
embedding_model = W.Text(value=EMBEDDING_MODEL, description="Embedding:", layout=W.Layout(width="650px"))
reranker_model = W.Text(value=RERANKER_MODEL, description="Reranker:", layout=W.Layout(width="650px"))
run_button = W.Button(description="Пересчитать", button_style="primary", icon="play")
progress = W.IntProgress(value=0, min=0, max=1, description="Новости:")
status = W.HTML()
run_output = W.Output()

def run_clustering(_=None):
    global clusters, run_info
    run_button.disabled = True
    with run_output:
        clear_output(wait=True)
        try:
            rows, embeddings, filter_counts = get_data_and_vectors(
                embedding_chars.value, embedding_batch_size.value, embedding_model.value.strip())
            progress.max, progress.value = len(rows), 0
            status.value = "Загрузка локального reranker…"
            reranker = LocalReranker(reranker_model.value.strip(), device=DEVICE)
            started = time.perf_counter()
            def update_progress(done, total):
                progress.value = done
                status.value = f"Обработано {done}/{total}"
            result = cluster_posts(rows, embeddings, reranker, threshold=cosine_threshold.value,
                                   window_hours=window_hours.value, retrieval_top_k=reranker_candidates.value,
                                   rerank_top_k=reranker_candidates.value,
                                   reranker_batch_size=reranker_batch_size.value, on_progress=update_progress)
            run_info = {"embedding_model": embedding_model.value.strip(),
                        "reranker_model": reranker_model.value.strip(),
                        "cosine_threshold": cosine_threshold.value, "window_hours": window_hours.value,
                        "retrieval_top_k": reranker_candidates.value,
                        "rerank_top_k": reranker_candidates.value,
                        "embedding_chars": embedding_chars.value,
                        "embedding_batch_size": embedding_batch_size.value,
                        "reranker_batch_size": reranker_batch_size.value,
                        "seconds": round(time.perf_counter() - started, 2),
                        "filter_counts": filter_counts, "start": START.isoformat(),
                        "end_exclusive": END.isoformat()}
            tag = hashlib.sha256(json.dumps(run_info, sort_keys=True).encode()).hexdigest()[:12]
            destination = CACHE_DIR / f"clusters_{tag}.json"
            destination.write_text(json.dumps({"parameters": run_info, "clusters": result},
                                              ensure_ascii=False, indent=2), encoding="utf-8")
            clusters = result
            status.value = f"Готово: {len(rows)} новостей → {len(clusters)} кластеров"
            print(f"Готово: {len(clusters)} кластеров за {run_info['seconds']:.1f} с")
            print("Результат:", destination)
        except Exception:
            status.value = "Пересчет остановлен из-за ошибки; подробности ниже."
            import traceback
            traceback.print_exc()
        finally:
            run_button.disabled = False

run_button.on_click(run_clustering)
display(W.VBox([W.HBox([cosine_threshold, window_hours]),
                W.HBox([embedding_chars, embedding_batch_size]),
                W.HBox([reranker_candidates, reranker_batch_size]),
                embedding_model, reranker_model, run_button, progress, status, run_output]))


## HTML-отчет

После пересчета нажмите **Создать HTML**. Файл автономный: поиск работает по новостям, источникам и меткам; кластеры можно сворачивать.


In [5]:
from core.cluster_report import export_html as _export_html

def export_html(path):
    return _export_html(path, clusters, run_info)

html_button=W.Button(description='Создать HTML',button_style='primary',icon='file-code')
html_output=W.Output()
def create_html(_):
    with html_output:
        clear_output(wait=True)
        try: print('HTML отчет:',export_html(OUTPUT/'cluster_explorer'/'clusters_browser.html'))
        except RuntimeError as exc: print(exc)
html_button.on_click(create_html)
display(W.VBox([html_button,html_output]))
